# Audit và reproducibility của phạm vi đã đóng

Không phải notebook training. Public-summary mode trình bày bằng chứng versioned; FULL_AUDIT cần originals riêng tư và dataset đúng hash, chạy audit/statistics CPU. Không tự tải raw predictions hoặc tạo missing experiments.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/course_final.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.course_final import analyze, ARCHIVE_NAME, ARCHIVE_SHA256
from src.analysis.master_session1 import BUNDLE_HASH
TABLES = ROOT / "results/tables/course_final"
FULL_AUDIT = False  # True requires three private source ZIPs and the matching dataset.
REBUILD_PUBLICATION = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_FINAL_ARCHIVE", str(ROOT / "artifacts/e2_final" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, 10000)
if REBUILD_PUBLICATION:
    from gen_fig_course_final import generate
    from write_course_final_report import write_report
    generate(ROOT)
    write_report(ROOT)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["runs_verified"] == 28 and audit["prior_changed_files"] == 0
assert audit["training_closed"] and audit["remaining_requested_training_runs"] == 0
print("Raw re-audit" if FULL_AUDIT else "Public audited-table mode, not a fresh raw audit")
print("Source:", ARCHIVE_SHA256)


In [ ]:
display(pd.Series(audit))
comparison = pd.read_csv(TABLES / "prior_run_comparison.csv")
assert len(comparison) == 262 and comparison.unchanged.all()
assert (comparison.old_sha256 == comparison.new_sha256).all()
display(comparison)
inventory = pd.read_csv(TABLES / "archive_inventory.csv")
print("Archive members:", len(inventory))
print("Prior logical bytes reused:", inventory.loc[inventory.deduplicated_prior_content, "bytes"].sum())


## Checkpoint index ghép, không nhân bản lại old weights

Release phiên 1 có 22 run; delta mới chỉ sáu run. CLI `python scripts/fetch_course_final_checkpoints.py` tải 28 best; thêm `--with-resume` lấy 27 latest states. Hash ZIP và từng tệp được kiểm tra. MostPopular là NPZ, không dùng PyTorch loader cho nó. Gói weights không thay thế ZIP nguyên gốc để re-audit per-user metrics.


In [ ]:
manifest = json.loads((ROOT / "configs/artifacts/course_final_checkpoints.json").read_text())
assert len(manifest["models"]) == manifest["best_checkpoints"] == 28
assert manifest["latest_states"] == 27
display(pd.DataFrame(manifest["models"]).T)
scope = json.loads((ROOT / "configs/experiments/course_jobs.json").read_text())
assert scope["training_closed"] and len(scope["jobs"]) == 28
print("No requested remaining training runs. Full 127-job grid is reference only.")


## Báo cáo và hình/bảng

Đọc [COURSE_FINAL_REPORT_VI.md](../docs/COURSE_FINAL_REPORT_VI.md), [phương pháp/limitations](../docs/EXPERIMENT_METHODS_REPRODUCIBILITY_VI.md) và [asset catalog](../docs/REPORT_ASSET_CATALOG.md). PNG 300 DPI cho slides, PDF vector cho report. Không đổi SD N/A thành 0, không nhầm SD với CI và không gán dedicated E7 COMPLETE.
